In [98]:
!pip install pypdf weaviate weaviate-client
!pip install -U langchain_community chromadb langchain langchain_openai
!pip install rank_bm25
!pip install --upgrade langchain
!pip install --upgrade langchain_community
!pip install --upgrade --quiet  weaviate-client
!pip install -U transformers accelerate pinecone
!pip install -Uqq langchain-weaviate
!pip install openai tiktoken
!pip install cohere langchain_cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.5 MB/s eta 0:00:00


In [97]:
import os
from google.colab import userdata
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA, RetrievalQAWithSourcesChain
import weaviate
from weaviate.classes.init import Auth
from langchain_weaviate.vectorstores import WeaviateVectorStore
import requests
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAI
from langchain.retrievers.document_compressors import CohereRerank
from langchain_cohere import CohereRerank

ModuleNotFoundError: No module named 'langchain_cohere'

In [93]:
OPENAI_API_TOKEN=userdata.get('OPENAI_API_KEY')
PINECONE_API_KEY=userdata.get('PINECONE_API_KEY')
COHERE_API_KEY = userdata.get('COHERE_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_TOKEN
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["COHERE_API_KEY"] = COHERE_API_KEY
WEAVIATE_URL="https://rvz2tdaetjehfxhvlbmiba.c0.us-west3.gcp.weaviate.cloud"
WEAVIATE_API_KEY="lFXOLsKv3rsQbuqvGlhTvOh6s04JxHciBQ34"

In [ ]:
doc_path = "/content/state_of_the_union.txt"

In [ ]:
embeddings = OpenAIEmbeddings()

<ipython-input-18-73ad2f8e367a>:1: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings()


In [ ]:
model = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
loder=TextLoader('state_of_the_union.txt', encoding="utf8")
document=loder.load()
text_spliiter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_spliiter.split_documents(document)

In [ ]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,  # Replace with your Weaviate Cloud URL
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY),  # Replace with your Weaviate Cloud key
    headers={'X-OpenAI-Api-key': OPENAI_API_TOKEN}  # Replace with your OpenAI API key
)

In [ ]:
client.is_ready()

True

In [ ]:
db = WeaviateVectorStore.from_documents(docs, embeddings, client=client)

In [ ]:
query = "What did the president say about Ketanji Brown Jackson"
docs = db.similarity_search(query)

# Print the first 100 characters of each result
for i, doc in enumerate(docs):
    print(f"\nDocument {i+1}:")
    print(doc.page_content[:100] + "...")


Document 1:
And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. On...

Document 2:
As I said last year, especially to our younger transgender Americans, I will always have your back a...

Document 3:
A former top litigator in private practice. A former federal public defender. And from a family of p...

Document 4:
But in my administration, the watchdogs have been welcomed back. 

We’re going after the criminals w...


In [105]:
#Hybrid Search using Weviate (Keyword: 0.8, Vector: 0.2 weightage)
docs = db.similarity_search(query, alpha=0.8)
docs[0]

Document(metadata={'source': 'state_of_the_union.txt'}, page_content='And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.')

In [106]:
docsearch = WeaviateVectorStore.from_documents(docs, embeddings, client=client, alpha=0.8)
hybrid_chain = RetrievalQA.from_chain_type(
    llm=model, chain_type="stuff", retriever=docsearch.as_retriever()
)

In [92]:
hybrid_response = hybrid_chain.invoke("What did the president say about Ketanji Brown Jackson")
hybrid_response

{'query': 'What did the president say about Ketanji Brown Jackson',
 'result': 'The president said that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson, describing her as one of the nation’s top legal minds who will continue Justice Breyer’s legacy of excellence.'}

In [100]:
cohere = CohereRerank()
compression_retriever = ContextualCompressionRetriever(
    base_compressor=cohere, base_retriever=docsearch.as_retriever()
)

In [102]:
compression_retriever.invoke("What did the president say about Ketanji Brown Jackson")

[Document(metadata={'source': 'state_of_the_union.txt', 'relevance_score': 0.7025301}, page_content='And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.'),
 Document(metadata={'source': 'state_of_the_union.txt', 'relevance_score': 0.05340333}, page_content='I know some are talking about “living with COVID-19”. Tonight – I say that we will never just accept living with COVID-19. \n\nWe will continue to combat the virus as we do other diseases. And because this is a virus that mutates and spreads, we will stay on guard. \n\nHere are four common sense steps as we move forward safely.'),
 Document(metadata={'source': 'state_of_the_union.txt', 'relevance_score': 0.01572399}, page_content='I understand. \n\nI remember when my Dad had to leave our home in Scranton, Pennsylvania to find work. I grew up in a family where if the price of food went up, you fe

In [103]:
hybrid_chain = RetrievalQA.from_chain_type(
    llm=model, chain_type="stuff", retriever=compression_retriever
)

In [104]:
hybrid_response = hybrid_chain.invoke("What did the president say about Ketanji Brown Jackson")
hybrid_response

{'query': 'What did the president say about Ketanji Brown Jackson',
 'result': 'The president referred to Circuit Court of Appeals Judge Ketanji Brown Jackson as one of the nation’s top legal minds and mentioned that she will continue Justice Breyer’s legacy of excellence.'}